# 11 — 二部图推荐系统与 LightGCN

本课把链接预测落到 MovieLens 推荐系统：根据用户过去喜欢的电影，为每个用户排列尚未看过的电影。我们比较热门度、矩阵分解和 LightGCN。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.recommendation import load_movielens_implicit, interaction_pairs, sample_bpr_negatives
from crosscity.models.recommendation import MatrixFactorization, build_lightgcn
from crosscity.training.recommendation import popularity_metrics, train_recommender
from crosscity.utils import seed_everything
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. 二部图是什么？

二部图 $G=(U\cup I,E)$ 的节点可以分成两个互不重叠的集合，边只允许跨集合出现。推荐系统中：

```text
用户集合 U                 电影集合 I
 user 0 ─────────────── movie 2
 user 0 ─────────────── movie 7
 user 1 ─────────────── movie 2
```

没有 user-user 或 movie-movie 交互边。一跳表示用户喜欢的电影；两跳 `用户→电影→用户` 找到兴趣相似的用户；三跳再到这些用户喜欢的其他电影。代码把电影节点整体偏移 `num_users`，从而在一个 `edge_index` 中获得不重叠的统一编号。

In [ ]:
data = load_movielens_implicit(ROOT / 'data/raw/movielens100k')
print('users/items/nodes:', data.num_users, data.num_items, data.num_nodes)
print('positive train/validation/test:', len(data.train_users),
      len(data.validation_users), len(data.test_users))
print('edge_index:', tuple(data.edge_index.shape),
      'user id range=', (int(data.edge_index[0].min()), data.num_users - 1),
      'movie node starts at=', data.num_users)
assert torch.all((data.edge_index[0] < data.num_users) !=
                 (data.edge_index[1] < data.num_users))

## 2. 显式评分如何变成隐式反馈？

MovieLens 原始标签是 1–5 分。本课把评分 ≥4 当成正反馈，只研究用户是否喜欢/交互，而不预测具体评分。没有评分的电影是未观察项，不一定是用户不喜欢。

训练期每个用户最近一次正反馈留给 validation；官方测试文件中的正反馈作为 test。LightGCN 的传播图只含 train 正边，因此不会提前看到目标交互。

## 3. 推荐中的负采样与 BPR

对每个正交互 $(u,i^+)$，从该用户没有正反馈的电影中抽取 $i^-$。BPR 不要求输出校准概率，而要求正电影排名更高：

$$\mathcal L_{BPR}=-\log\sigma(\hat y_{ui^+}-\hat y_{ui^-}).$$

如果正电影已经高于负电影，差值为正、损失较小；排序相反则损失较大。这里的负样本仍是未观察电影，而不是已知的讨厌电影。

In [ ]:
generator = torch.Generator().manual_seed(42)
example_users = data.train_users[:8]
example_positive = data.train_items[:8]
example_negative = sample_bpr_negatives(example_users, data.num_items, interaction_pairs(data),
                                        generator=generator)
pd.DataFrame({'user': example_users, 'positive movie': example_positive,
              'sampled negative movie': example_negative})

## 4. 三个模型分别学到什么？

1. Popularity：所有用户都收到相同的热门电影；
2. Matrix Factorization：直接学习用户向量 $p_u$ 和电影向量 $q_i$，分数为 $p_u^Tq_i$；
3. LightGCN：让嵌入在二部图上进行多层线性传播。

$$e_i^{(k+1)}=\sum_{j\in\mathcal N(i)}\frac{1}{\sqrt{d_i d_j}}e_j^{(k)},\qquad e_i=\sum_{k=0}^K\alpha_k e_i^{(k)}.$$

LightGCN 刻意删除普通 GCN 的特征变换和激活函数，保留协同过滤最需要的邻居传播。

## 5. Recall@K 与 NDCG@K

模型需要从所有未见电影中给出前 K 名。Recall@K 衡量真实喜欢电影有多少进入前 K；NDCG@K 进一步奖励把命中项排在更靠前的位置。评价前必须屏蔽训练中已经看过的电影，否则模型推荐旧电影也会获得虚假高分。

In [ ]:
print('Popularity validation@20:', popularity_metrics(data, split='validation', k=20))
print('Popularity test@20:', popularity_metrics(data, split='test', k=20))
for name, model in [('MF', MatrixFactorization(data.num_users, data.num_items, 32)),
                    ('LightGCN', build_lightgcn(data.num_users, data.num_items, 32, 3))]:
    seed_everything(42)
    result = train_recommender(model, data, device=device, max_epochs=3, patience=3, k=20)
    print(name, 'val=', result.validation, 'test=', result.test)

## 6. 三种子正式比较

数据切分和候选模型固定，只根据 validation NDCG@20 早停并恢复最佳 checkpoint。Popularity 没有随机参数，只运行一次。

In [ ]:
records, histories = [], {}
for seed in [42, 43, 44]:
    builders = {
        'MF': lambda: MatrixFactorization(data.num_users, data.num_items, 64),
        'LightGCN': lambda: build_lightgcn(data.num_users, data.num_items, 64, 3),
    }
    for name, builder in builders.items():
        seed_everything(seed)
        result = train_recommender(builder(), data, device=device, max_epochs=200,
                                   patience=20, k=20, seed=seed)
        records.append({'seed': seed, 'model': name, 'best_epoch': result.best_epoch,
                        'validation_recall@20': result.validation.recall,
                        'validation_ndcg@20': result.validation.ndcg,
                        'test_recall@20': result.test.recall,
                        'test_ndcg@20': result.test.ndcg})
        histories[(seed, name)] = result.history
results = pd.DataFrame(records)
display(results.sort_values(['seed', 'validation_ndcg@20'], ascending=[True, False]))
results.groupby('model')[['validation_recall@20', 'validation_ndcg@20',
                          'test_recall@20', 'test_ndcg@20']].agg(['mean', 'std']).style.format('{:.4f}')

In [ ]:
for (seed, name), history in histories.items():
    if seed == 42:
        frame = pd.DataFrame(history)
        plt.plot(frame.epoch, frame.val_ndcg, label=name)
plt.xlabel('epoch'); plt.ylabel('validation NDCG@20'); plt.legend(); plt.show()

## 7. 解释清单

1. MF 是否超过 Popularity？这代表个性化是否有效。
2. LightGCN 是否超过 MF？这代表图传播是否提供额外协同信号。
3. Recall 与 NDCG 的排序是否一致？如果不一致，模型改善的是覆盖还是顶部排序？
4. 随机负采样会不会总抽到太容易的电影？hard negative 有什么风险？
5. 没有历史交互的新用户和新电影为什么无法仅靠 LightGCN 推荐？这就是 cold start。